In [15]:
from ucimlrepo import fetch_ucirepo

steel_plates_faults = fetch_ucirepo(id=198)

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import numpy as np

device = "cuda" if torch.cuda.is_available() else "cpu"

X = steel_plates_faults.data.features
y = steel_plates_faults.data.targets

# y jest one-hot encoded (7 kolumn) → zamieniamy na indeksy klas (0-6)
y = np.argmax(y.values, axis=1)          # lub y.to_numpy()

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Konwersja na tensory PyTorch
X_train_tensor = torch.from_numpy(X_train_scaled).float().to(device)
X_test_tensor = torch.from_numpy(X_test_scaled).float().to(device)
y_train_tensor = torch.from_numpy(y_train).long().to(device)
y_test_tensor = torch.from_numpy(y_test).long().to(device)

train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

# ==================== MODEL Z DOKŁADNIE 3 WARSTWAMI + Softmax ====================
class SteelFaultNet(nn.Module):
    def __init__(self):
        super(SteelFaultNet, self).__init__()
        # self.fc1 = nn.Linear(27, 128)   # 27 cech wejściowych
        # self.fc2 = nn.Linear(128, 64)
        # self.fc3 = nn.Linear(64, 7)     # 7 klas
        self.fc1 = nn.Linear(27, 64) # 27 cech wejściowych
        self.fc2 = nn.Linear(64, 16)
        # nn.BatchNorm1d(27) 
        self.fc3 = nn.Linear(16, 7)  
        
        
    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.fc3(x)
        # x = F.softmax(x, dim=1)         # jawnie zadeklarowany Softmax
        return x


model = SteelFaultNet().to(device)

criterion = nn.CrossEntropyLoss()                    # działa z logitami, ale skoro masz softmax, można też użyć NLLLoss
optimizer = optim.Adam(model.parameters(), lr=0.001)

# ==================== TRENING ====================
epochs = 20

for epoch in range(epochs):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for x_batch, y_batch in train_loader:
        optimizer.zero_grad()
        outputs = model(x_batch)
        loss = criterion(outputs, y_batch)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        
        _, predicted = torch.max(outputs, 1)
        total += y_batch.size(0)
        correct += (predicted == y_batch).sum().item()

    avg_loss = running_loss / len(train_loader)
    train_acc = 100 * correct / total
    print(f"Epoch {epoch+1:2d} | Loss: {avg_loss:.4f} | Train Acc: {train_acc:.2f}%")

# ==================== TEST ====================
with torch.no_grad():
    model.eval()
    outputs = model(X_test_tensor)
    loss = criterion(outputs, y_test_tensor).item()
    
    _, predicted = torch.max(outputs, 1)
    accuracy = (predicted == y_test_tensor).float().mean().item() * 100

print(f"\nTest Loss: {loss:.4f}")
print(f"Test Accuracy: {accuracy:.2f}%")

Epoch  1 | Loss: 1.7518 | Train Acc: 41.49%
Epoch  2 | Loss: 1.2634 | Train Acc: 55.03%
Epoch  3 | Loss: 1.0554 | Train Acc: 58.70%
Epoch  4 | Loss: 0.9218 | Train Acc: 65.72%
Epoch  5 | Loss: 0.8255 | Train Acc: 68.81%
Epoch  6 | Loss: 0.7651 | Train Acc: 72.29%
Epoch  7 | Loss: 0.7214 | Train Acc: 72.62%
Epoch  8 | Loss: 0.6848 | Train Acc: 73.71%
Epoch  9 | Loss: 0.6613 | Train Acc: 73.90%
Epoch 10 | Loss: 0.6452 | Train Acc: 75.39%
Epoch 11 | Loss: 0.6289 | Train Acc: 75.84%
Epoch 12 | Loss: 0.6147 | Train Acc: 76.29%
Epoch 13 | Loss: 0.6016 | Train Acc: 76.87%
Epoch 14 | Loss: 0.5829 | Train Acc: 76.68%
Epoch 15 | Loss: 0.5768 | Train Acc: 77.77%
Epoch 16 | Loss: 0.5611 | Train Acc: 77.58%
Epoch 17 | Loss: 0.5599 | Train Acc: 77.51%
Epoch 18 | Loss: 0.5477 | Train Acc: 77.84%
Epoch 19 | Loss: 0.5405 | Train Acc: 78.29%
Epoch 20 | Loss: 0.5291 | Train Acc: 77.96%

Test Loss: 0.6690
Test Accuracy: 75.58%
